# Decrypting Ciphers Using Markov Chain Monte Carlo — A Visual Walkthrough
---

We combine two ideas — Markov chains converging to a stationary distribution and Monte Carlo estimation — to crack a substitution cipher on a Spanish text from Borges' *Ficciones*.

**Outline:**
1. Markov Chain Simulation & Stationary Distribution
2. Building a Bigram Model (PyTorch)
3. Substitution Cipher Functions
4. Scoring Function
5. The Metropolis Algorithm
6. Convergence Visualization

This notebook creates all plots referenced in the companion blogpost.

---

## 0. Setup

In [ ]:
!pip install -q scienceplots

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import scienceplots
import random
import math
import unicodedata
import warnings
warnings.filterwarnings('ignore')

plt.style.use(["science", "ieee", "no-latex"])
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# Output directory for images
IMG_DIR = './'

print("Ready to go!")

---
## 1. Markov Chain Simulation & Stationary Distribution

We simulate a 3-state Markov chain (A, B, C) and show that the empirical visit frequencies converge to the stationary distribution regardless of starting state.

In [ ]:
# Transition matrix
P_mc = np.array([
    [0.2, 0.5, 0.3],  # from A
    [0.1, 0.6, 0.3],  # from B
    [0.4, 0.2, 0.4],  # from C
])

states = ['A', 'B', 'C']
n_steps = 5000

def simulate_chain(P, start_state, n_steps):
    """Simulate a Markov chain and return the sequence of states."""
    n_states = P.shape[0]
    chain = [start_state]
    for _ in range(n_steps - 1):
        current = chain[-1]
        next_state = np.random.choice(n_states, p=P[current])
        chain.append(next_state)
    return np.array(chain)

# Compute true stationary distribution (left eigenvector with eigenvalue 1)
eigenvalues, eigenvectors = np.linalg.eig(P_mc.T)
idx = np.argmin(np.abs(eigenvalues - 1.0))
pi_true = np.real(eigenvectors[:, idx])
pi_true = pi_true / pi_true.sum()
print(f"True stationary distribution: π = ({pi_true[0]:.4f}, {pi_true[1]:.4f}, {pi_true[2]:.4f})")

In [ ]:
# Also verify convergence by matrix power
print("Convergence of s_t = s_0 · P^t (starting from A):")
print(f"{'Step':>6}  {'P(A)':>8}  {'P(B)':>8}  {'P(C)':>8}")
s = np.array([1.0, 0.0, 0.0])
for step in [0, 1, 2, 5, 10, 50]:
    s_t = s @ np.linalg.matrix_power(P_mc, step)
    print(f"{step:>6}  {s_t[0]:>8.4f}  {s_t[1]:>8.4f}  {s_t[2]:>8.4f}")

In [ ]:
# Simulate from all 3 starting states and plot convergence of empirical frequencies
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
colors = ['#e74c3c', '#3498db', '#2ecc71']

for start_idx, ax in enumerate(axes):
    chain = simulate_chain(P_mc, start_idx, n_steps)
    
    # Compute running proportions
    steps_range = np.arange(1, n_steps + 1)
    for state_idx in range(3):
        cumulative = np.cumsum(chain == state_idx)
        proportions = cumulative / steps_range
        ax.plot(steps_range, proportions, color=colors[state_idx], 
                label=f'{states[state_idx]}', lw=1.2, alpha=0.85)
        ax.axhline(y=pi_true[state_idx], color=colors[state_idx], 
                   ls='--', alpha=0.5, lw=0.8)
    
    ax.set_title(f'Start: {states[start_idx]}', fontsize=11)
    ax.set_xlabel('Step')
    if start_idx == 0:
        ax.set_ylabel('Visit proportion')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(-0.05, 1.05)

fig.suptitle('Markov Chain Convergence to Stationary Distribution', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}mcmc-markov-chain-simulation.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: mcmc-markov-chain-simulation.png")

---
## 2. Building a Bigram Model with PyTorch

We build a character-level bigram frequency matrix from a Spanish corpus. Since we may not have a large corpus file handy, we'll generate one from a substantial Spanish text embedded here.

The alphabet is 27 characters: `a-z` + space (normalized Spanish — no `ñ`, no accents, no punctuation).

In [ ]:
# --- Spanish corpus ---
# Read corpus from a text file
with open('corpus.txt', 'r', encoding='utf-8') as f:
    spanish_corpus_raw = f.read()

print(f"Raw corpus length: {len(spanish_corpus_raw)} characters")

In [ ]:
# Normalize: lowercase, remove accents, ñ -> n, keep only a-z and space
def normalize_spanish(s):
    s = s.lower()
    s = s.replace('\n', ' ')  # treat newlines as word boundaries
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')  # strip accents
    s = s.replace('ñ', 'n')
    s = ''.join(c if c in 'abcdefghijklmnopqrstuvwxyz ' else '' for c in s)
    # Collapse multiple spaces
    s = ' '.join(s.split())
    return s

corpus = normalize_spanish(spanish_corpus_raw)
print(f"Normalized corpus length: {len(corpus)} characters")
print(f"First 200 chars: {corpus[:200]}")

In [ ]:
# Build bigram count matrix using PyTorch
# Character-to-index mapping: '.' = 0 (boundary), then ' '=1, a=2, ..., z=27
words = corpus.split()

chars = sorted(list(set(' '.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

print(f"Number of unique characters: {len(chars)}")
print(f"Character mapping: {stoi}")
print(f"Matrix size: {len(stoi)} x {len(stoi)}")

In [ ]:
# Count bigrams over continuous text (including spaces)
n_tokens = len(stoi)
N = torch.zeros((n_tokens, n_tokens), dtype=torch.int32)

# Use '.' as boundary at start/end of entire corpus
chs = ['.'] + list(corpus) + ['.']
for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1

print(f"Total bigram counts: {N.sum().item()}")
print(f"Most common bigrams (top 10):")

# Find top bigrams
flat = N.flatten()
top_indices = torch.argsort(flat, descending=True)[:10]
for idx in top_indices:
    i, j = idx // n_tokens, idx % n_tokens
    ch1 = itos[i.item()]
    ch2 = itos[j.item()]
    count = N[i, j].item()
    print(f"  '{ch1}{ch2}': {count}")

In [ ]:
# Convert to probability matrix with Laplace smoothing
P = (N + 1).float()
P /= P.sum(1, keepdims=True)

print(f"Row sums (should all be ~1.0): {P.sum(1)[:5]}")
print(f"P('d' -> 'e') = {P[stoi['d'], stoi['e']].item():.4f}")
print(f"P('q' -> 'z') = {P[stoi['q'], stoi['z']].item():.6f}")

In [ ]:
# Visualize the bigram matrix
fig, ax = plt.subplots(figsize=(12, 11))

# Use log scale for better visualization (add 1 to handle zeros)
n_chars = len(stoi)  # total vocab size including boundary token
N_display = N[1:, 1:].float()  # exclude boundary token for display
display_size = n_chars - 1
display_chars = [itos[i] if itos[i] != ' ' else '⎵' for i in range(1, n_chars)]

im = ax.imshow(N_display.numpy(), cmap='Blues', aspect='equal')

# Add character labels
ax.set_xticks(range(display_size))
ax.set_yticks(range(display_size))
ax.set_xticklabels(display_chars, fontsize=8)
ax.set_yticklabels(display_chars, fontsize=8)
ax.set_xlabel('Next character', fontsize=11)
ax.set_ylabel('Current character', fontsize=11)
ax.set_title('Bigram Count Matrix — Spanish Corpus', fontsize=12)

# Add count annotations for high-count cells
for i in range(display_size):
    for j in range(display_size):
        val = int(N_display[i, j].item())
        if val > 30:
            ax.text(j, i, str(val), ha='center', va='center', 
                    fontsize=5, color='white' if val > 100 else 'gray')

plt.colorbar(im, ax=ax, shrink=0.8, label='Count')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}mcmc-bigram-heatmap.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: mcmc-bigram-heatmap.png")

---
## 3. Substitution Cipher Functions

We define our cipher over the 27-character normalized Spanish alphabet: `a-z` + space.

In [ ]:
# Alphabet: a-z + space (27 characters)
alphabet = list('abcdefghijklmnopqrstuvwxyz ')

def generate_cipher():
    """Generate a random permutation of the alphabet."""
    cipher = alphabet.copy()
    random.shuffle(cipher)
    return cipher

def encode(text, cipher):
    """Encode text using a cipher (permutation)."""
    mapping = {original: coded for original, coded in zip(alphabet, cipher)}
    return ''.join(mapping.get(c, c) for c in text)

def decode(ciphertext, cipher):
    """Decode ciphertext using a cipher (inverse permutation)."""
    mapping = {coded: original for original, coded in zip(alphabet, cipher)}
    return ''.join(mapping.get(c, c) for c in ciphertext)

def swap_two(cipher):
    """Propose a new cipher by swapping two random positions."""
    new_cipher = cipher.copy()
    i, j = random.sample(range(len(cipher)), 2)
    new_cipher[i], new_cipher[j] = new_cipher[j], new_cipher[i]
    return new_cipher

print(f"Alphabet size: {len(alphabet)}")
print(f"Search space: {len(alphabet)}! ≈ {math.factorial(len(alphabet)):.2e} permutations")

In [ ]:
# The Borges plaintext
plaintext = ("he dicho que los hombres de ese planeta conciben el universo "
             "como una serie de procesos mentales que no se desenvuelven "
             "en el espacio sino de modo sucesivo en el tiempo")

print(f"Plaintext ({len(plaintext)} chars):")
print(plaintext)
print()

# Test encode/decode roundtrip
test_cipher = generate_cipher()
ciphertext_test = encode(plaintext, test_cipher)
decoded_test = decode(ciphertext_test, test_cipher)

print(f"Cipher: {''.join(test_cipher)}")
print(f"Ciphertext: {ciphertext_test}")
print(f"Decoded:    {decoded_test}")
print(f"Roundtrip OK: {decoded_test == plaintext}")

---
## 4. Scoring Function

We score a candidate decryption by its **bigram log-likelihood** under our Spanish bigram model.

In [ ]:
def score_text(text, P, stoi):
    """Compute log-likelihood of a text under the bigram model."""
    log_prob = 0.0
    for ch1, ch2 in zip(text, text[1:]):
        ix1 = stoi.get(ch1, 0)
        ix2 = stoi.get(ch2, 0)
        log_prob += torch.log(P[ix1, ix2]).item()
    return log_prob

# Validate: Spanish text should score much higher than gibberish
spanish_score = score_text("esto es texto en espanol", P, stoi)
gibberish_score = score_text("fghr gh wghdfrf etfs xqzk", P, stoi)

print(f"Spanish text score:  {spanish_score:.2f}")
print(f"Gibberish score:     {gibberish_score:.2f}")
print(f"Difference:          {spanish_score - gibberish_score:.2f} (positive = Spanish wins)")
print()

# Also test on the actual plaintext vs its ciphertext
plaintext_score = score_text(plaintext, P, stoi)
ciphertext_score = score_text(ciphertext_test, P, stoi)
print(f"Borges plaintext score:  {plaintext_score:.2f}")
print(f"Borges ciphertext score: {ciphertext_score:.2f}")

---
## 5. The Metropolis Algorithm

We run the Metropolis algorithm to decode the ciphertext. The chain proposes small changes (letter swaps) and accepts/rejects based on the bigram log-likelihood ratio.

In [ ]:
# Set up the cipher problem
true_cipher = generate_cipher()
ciphertext = encode(plaintext, true_cipher)

print(f"True cipher: {''.join(true_cipher)}")
print(f"Ciphertext:  {ciphertext}")
print(f"Verify decode: {decode(ciphertext, true_cipher) == plaintext}")

In [ ]:
# Run the Metropolis algorithm
n_iterations = 30000

# Start with a random cipher
current_cipher = generate_cipher()
current_decoded = decode(ciphertext, current_cipher)
current_score = score_text(current_decoded, P, stoi)

# Track history for visualization
score_history = [current_score]
decoded_history = [(0, current_decoded)]
accept_count = 0
best_score = current_score
best_decoded = current_decoded
best_cipher = current_cipher.copy()

for iteration in range(1, n_iterations + 1):
    # Propose a new cipher by swapping two letters
    proposed_cipher = swap_two(current_cipher)
    
    # Score the proposed decryption
    proposed_decoded = decode(ciphertext, proposed_cipher)
    proposed_score = score_text(proposed_decoded, P, stoi)
    
    # Acceptance probability (log scale)
    log_alpha = proposed_score - current_score
    
    # Accept or reject
    if log_alpha >= 0 or random.random() < math.exp(log_alpha):
        current_cipher = proposed_cipher
        current_score = proposed_score
        current_decoded = proposed_decoded
        accept_count += 1
        
        # Track best
        if current_score > best_score:
            best_score = current_score
            best_decoded = current_decoded
            best_cipher = current_cipher.copy()
    
    score_history.append(current_score)
    
    # Log selected iterations
    if iteration in [1, 5, 10, 50, 100, 500, 1000, 2000, 5000, 10000, 20000, 30000]:
        decoded_history.append((iteration, current_decoded))

print(f"Acceptance rate: {accept_count / n_iterations:.2%}")
print(f"\nBest score: {best_score:.2f}")
print(f"Best decoded text:")
print(f"  {best_decoded}")
print(f"\nOriginal plaintext:")
print(f"  {plaintext}")
print(f"\nMatch: {best_decoded == plaintext}")

In [ ]:
# Print convergence trace
print("Decoded text over iterations:")
print("=" * 90)
for iteration, text in decoded_history:
    # Truncate for display
    display = text[:80] + ('...' if len(text) > 80 else '')
    print(f"Iter {iteration:>6}: {display}")

---
## 6. Convergence Visualization

We create two plots:
1. **Cipher convergence**: showing the decoded text at key iterations
2. **Burn-in score**: the log-likelihood over iterations

In [ ]:
# Plot 1: Cipher convergence — show decoded text at selected iterations
# Select a subset of iterations to display
display_iters = [i for i in [0, 1, 10, 50, 100, 500, 1000, 2000, 5000, 10000, 20000, 30000]
                 if any(it == i for it, _ in decoded_history)]
display_entries = [(it, txt) for it, txt in decoded_history if it in display_iters]

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_axis_off()

n_rows = len(display_entries)
y_positions = np.linspace(0.95, 0.05, n_rows)

ax.set_title('Metropolis Algorithm: Decoding a Substitution Cipher', fontsize=13, pad=15)

for idx, (iteration, text) in enumerate(display_entries):
    y = y_positions[idx]
    # Truncate text for display
    display_text = text[:70] + ('...' if len(text) > 70 else '')
    
    # Color: red for early (gibberish), green for converged
    score_at_iter = score_history[min(iteration, len(score_history)-1)]
    # Normalize color based on score range
    score_range = max(score_history) - min(score_history)
    if score_range > 0:
        t = (score_at_iter - min(score_history)) / score_range
    else:
        t = 0.5
    color = plt.cm.RdYlGn(t)
    
    ax.text(0.0, y, f'Iter {iteration:>6}:', fontsize=8, fontfamily='monospace',
            fontweight='bold', transform=ax.transAxes, va='center')
    ax.text(0.14, y, display_text, fontsize=7.5, fontfamily='monospace',
            color=color, transform=ax.transAxes, va='center')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}mcmc-cipher-convergence.png', dpi=200, bbox_inches='tight',
            facecolor='white')
plt.show()
print("Saved: mcmc-cipher-convergence.png")

In [ ]:
# Plot 2: Burn-in — log-likelihood score over iterations
fig, ax = plt.subplots(figsize=(10, 4))

iterations = np.arange(len(score_history))
ax.plot(iterations, score_history, lw=0.5, alpha=0.7, color='steelblue')

# Add smoothed version
window = 200
if len(score_history) > window:
    smoothed = np.convolve(score_history, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window-1, window-1+len(smoothed)), smoothed, 
            lw=2, color='#e74c3c', label=f'Moving avg (w={window})')

# Mark the plaintext score for reference
true_score = score_text(plaintext, P, stoi)
ax.axhline(y=true_score, color='#2ecc71', ls='--', lw=1.5, alpha=0.7,
           label=f'True plaintext score ({true_score:.1f})')

# Mark burn-in region
ax.axvspan(0, 2000, alpha=0.08, color='red', label='Burn-in region')

ax.set_xlabel('Iteration')
ax.set_ylabel('Log-likelihood Score')
ax.set_title('Metropolis Algorithm: Score Convergence (Burn-in)')
ax.legend(fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}mcmc-burnin-score.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: mcmc-burnin-score.png")

---
## 7. Summary

This notebook demonstrated:

1. **Markov chain convergence** — empirical visit frequencies converge to the stationary distribution $\pi$ regardless of starting state
2. **Bigram model** — character-level transition probabilities from a Spanish corpus, built with PyTorch
3. **Substitution cipher** — encoding/decoding over a 27-character normalized Spanish alphabet
4. **Log-likelihood scoring** — bigram scores that differentiate Spanish from gibberish
5. **Metropolis algorithm** — iterative letter-swap proposals with accept/reject, converging from random gibberish to the Borges plaintext